# 03 — Model Benchmark

Train and compare Prophet, XGBoost, LSTM, and Ensemble models.
Evaluate on the validation and test sets using MAPE, RMSE, and MAE.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_processed
from src.features import (
    create_feature_pipeline, train_val_test_split, get_feature_columns
)
from src.train import train_ensemble, predict_ensemble
from src.utils import (
    compute_metrics, plot_forecast_vs_actual, plot_residuals, save_model
)
from src.config import CATEGORY_COL, TARGET_COL, MODELS_DIR

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12})
%matplotlib inline

print('Libraries loaded.')

In [ ]:
# Load and prepare data
daily = load_processed('daily_category_demand')
featured = create_feature_pipeline(daily)
train, val, test = train_val_test_split(featured)
cat_features = get_feature_columns(featured)

print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')
print(f'Features: {len(cat_features)}')

## Train Models for Top Categories

In [ ]:
# Select top N categories by volume
top_cats = (
    train.groupby(CATEGORY_COL)[TARGET_COL]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
)
print(f'Top categories: {top_cats}')

In [ ]:
all_results = {}
for cat in top_cats:
    print(f'\n{"="*50}')
    print(f'Training: {cat}')
    print(f'{"="*50}')
    
    results = train_ensemble(train, val, cat_features, cat)
    all_results[cat] = results
    
    # Test predictions
    y_pred = predict_ensemble(results, val, test, cat_features, cat)
    y_true = test[test[CATEGORY_COL] == cat][TARGET_COL].values
    
    if len(y_true) == len(y_pred) and len(y_true) > 0:
        test_metrics = compute_metrics(y_true, y_pred)
        all_results[cat]['test_metrics'] = test_metrics
        print(f'\n  Test MAPE: {test_metrics["mape"]:.2f}%')
        print(f'  Test RMSE: {test_metrics["rmse"]:.2f}')
        print(f'  Test MAE:  {test_metrics["mae"]:.2f}')
    
    # Save model
    save_model(results, f'ensemble_{cat}.pkl')

## Results Summary

In [ ]:
results_summary = []
for cat, res in all_results.items():
    if 'test_metrics' in res:
        row = {'category': cat}
        row.update(res['test_metrics'])
        results_summary.append(row)

results_df = pd.DataFrame(results_summary)
print('Test Results:')
results_df

In [ ]:
# Aggregate
print(f'Average MAPE: {results_df["mape"].mean():.2f}%')
print(f'Average RMSE: {results_df["rmse"].mean():.2f}')

## Forecast vs Actual Plots

In [ ]:
for cat in top_cats[:3]:
    if cat not in all_results:
        continue
    results = all_results[cat]
    
    y_pred = predict_ensemble(results, val, test, cat_features, cat)
    y_true = test[test[CATEGORY_COL] == cat][TARGET_COL].values
    dates = test[test[CATEGORY_COL] == cat]['date'].values
    
    if len(y_true) == len(y_pred):
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(dates, y_true, label='Actual', linewidth=1.5, color='black')
        ax.plot(dates, y_pred, label='Ensemble Forecast', linewidth=1.2, color='#2E86AB')
        ax.fill_between(dates, y_pred, y_true, alpha=0.15, color='red')
        ax.set_title(f'Ensemble Forecast vs Actual — {cat}')
        ax.legend()
        plt.tight_layout()
        plt.show()

## Residual Analysis

In [ ]:
cat = top_cats[0]
results = all_results[cat]
y_pred = predict_ensemble(results, val, test, cat_features, cat)
y_true = test[test[CATEGORY_COL] == cat][TARGET_COL].values

if len(y_true) == len(y_pred):
    residuals = y_true - y_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].scatter(range(len(residuals)), residuals, alpha=0.5, s=10)
    axes[0].axhline(y=0, color='red', linestyle='--')
    axes[0].set_title(f'Residuals — {cat}')
    axes[0].set_xlabel('Sample')
    axes[0].set_ylabel('Residual')
    
    axes[1].hist(residuals, bins=20, edgecolor='white', alpha=0.7)
    axes[1].axvline(x=0, color='red', linestyle='--')
    axes[1].set_title(f'Residual Distribution — {cat}')
    axes[1].set_xlabel('Residual')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Residual stats:')
    print(f'  Mean: {residuals.mean():.2f}')
    print(f'  Std:  {residuals.std():.2f}')

## Model Weights

In [ ]:
for cat in top_cats:
    if cat in all_results and 'ensemble_weights' in all_results[cat]:
        weights = all_results[cat]['ensemble_weights']
        print(f'{cat:25s} → ', end='')
        for name, w in sorted(weights.items()):
            print(f'{name}={w:.2f}  ', end='')
        print()